# Notebook 1 — Qwen 3.5 9B: confidence-guided FullDist + Ranked Nodes — Google Colab

**Objective:** generate CPTs without consulting the `bnRep` reference, using a hybrid strategy to reduce LLM calls.

This version is designed for **Google Colab with a GPU** and runs four independent conditions:

1. control without a persona;
2. structural engineer;
3. seismologist;
4. risk analyst.

The protocol removes `SepState`: it uses **FullDist for rows where the LLM reports sufficient confidence** and **Ranked Nodes for the remaining eligible rows**.

**This version neither mounts nor requests access to Google Drive.**  
Manually upload the two JSON files to the Colab environment; results are stored in `/content/`.

**Compatibility:** runtime messages and input/output conventions remain in Portuguese. Public configuration names and model prompts are unchanged.


## 0. Set up Google Colab

Before running, select **Runtime > Change runtime type > GPU**.

The next cell installs the required dependencies. On common Colab GPUs, such as T4 or L4, the notebook **automatically enables 4-bit NF4** to fit in VRAM. On GPUs with more memory, such as A100, it retains BF16/FP16 by default.

> For a direct scientific comparison with a previous run, keep the same precision/quantization across models or record the difference as part of the experimental configuration.


In [ ]:
# ============================================================
# INSTALL DEPENDENCIES IN GOOGLE COLAB
# Run this cell before the others.
# ============================================================
import sys, subprocess, importlib.util, os
from pathlib import Path
os.environ.setdefault('HF_HOME','/content/hf_cache')
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)

packages = [
    "git+https://github.com/huggingface/transformers.git",
    "accelerate>=1.0.0",
    "bitsandbytes>=0.46.1",
    "huggingface_hub>=0.25.0",
    "pgmpy>=1.0.0",
    "scipy>=1.11.0",
    "networkx>=3.2",
    "pandas>=2.0.0",
]

print("[COLAB] Instalando/atualizando dependências...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", *packages
])

required_modules = (
    "torch", "transformers", "accelerate", "bitsandbytes",
    "huggingface_hub", "pgmpy", "numpy", "networkx", "scipy", "pandas"
)
missing_modules = [n for n in required_modules if importlib.util.find_spec(n) is None]
if missing_modules:
    raise RuntimeError("Bibliotecas faltantes após a instalação: " + ", ".join(missing_modules))

import transformers
print("[DEPENDÊNCIAS] OK | transformers =", transformers.__version__)

## 0.1. Input files — no Google Drive

Manually upload both files through the Google Colab **Files** panel:

- Bayesian network JSON;
- `personas.json`.

By default, the notebook looks for:

- `/content/rede.json`
- `/content/personas.json`

If your files have different names, change only the two paths in the cell below.

**No Google Drive access is requested.**


In [ ]:
# ============================================================
# LOCAL COLAB FILES — NO GOOGLE DRIVE
# ============================================================
from pathlib import Path
import os

# 1) Manually upload the files through the Colab "Files" panel.
# 2) Adjust only these paths if the file names differ.
CAMINHO_REDE_MANUAL = Path('/content/rede.json')
CAMINHO_PERSONAS_MANUAL = Path('/content/personas.json')

if not CAMINHO_REDE_MANUAL.is_file():
    raise FileNotFoundError(
        f'JSON da rede não encontrado em {CAMINHO_REDE_MANUAL}. '
        'Envie o arquivo manualmente para /content ou ajuste CAMINHO_REDE_MANUAL.'
    )

if not CAMINHO_PERSONAS_MANUAL.is_file():
    raise FileNotFoundError(
        f'JSON de personas não encontrado em {CAMINHO_PERSONAS_MANUAL}. '
        'Envie o arquivo manualmente para /content ou ajuste CAMINHO_PERSONAS_MANUAL.'
    )

# Everything stays local to the Colab runtime.
PASTA_PROJETO = Path('/content/Qwen_BN_Confidence_RN')
PASTA_PROJETO.mkdir(parents=True, exist_ok=True)

os.environ['EXPERIMENT_BASE_DIR'] = str(PASTA_PROJETO)
os.environ['REDE_JSON'] = str(CAMINHO_REDE_MANUAL)
os.environ['PERSONAS_JSON'] = str(CAMINHO_PERSONAS_MANUAL)
os.environ['HF_HOME'] = '/content/hf_cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)

print('Rede:', CAMINHO_REDE_MANUAL)
print('Personas:', CAMINHO_PERSONAS_MANUAL)
print('Projeto/saídas:', PASTA_PROJETO)
print('Cache do modelo:', os.environ['HF_HOME'])


In [ ]:
# ============================================================
# MAIN CONFIGURATION — GOOGLE COLAB
# ============================================================
import os, json, time, re, itertools, hashlib, random, platform
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime, timezone
import torch

COLAB_MODE = True
BASE_DIR = Path(os.environ.get('EXPERIMENT_BASE_DIR', '/content/Qwen_BN_Confidence_RN'))
BASE_DIR.mkdir(parents=True, exist_ok=True)

CAMINHO_JSON = Path(os.environ.get('REDE_JSON', str(BASE_DIR/'entradas'/'rede.json')))
CAMINHO_PERSONAS = Path(os.environ.get('PERSONAS_JSON', str(BASE_DIR/'entradas'/'personas.json')))

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU CUDA não encontrada. No Colab, escolha: Ambiente de execução > '
        'Alterar tipo de ambiente de execução > GPU.'
    )

_gpu = torch.cuda.get_device_properties(0)
GPU_VRAM_GB = _gpu.total_memory / 2**30
print(f'[GPU] {_gpu.name} | VRAM: {GPU_VRAM_GB:.1f} GiB')

HF_MODEL_ID = 'Qwen/Qwen3.5-9B'
QWEN35_SEM_THINKING = True
HF_MODEL_REVISION = os.environ.get('HF_MODEL_REVISION', 'main')
PERGUNTAR_FAMILIARIDADE_REDE = os.environ.get('PERGUNTAR_FAMILIARIDADE_REDE','1').lower() not in ('0','false','no')

NOME_REDE_BNREP = os.environ.get('BNREP_REDE', 'seismic')
if CAMINHO_JSON.is_file() and not os.environ.get('BNREP_REDE'):
    _network_metadata = json.loads(CAMINHO_JSON.read_text(encoding='utf-8-sig'))
    _network_info = _network_metadata.get('network', {})
    if isinstance(_network_info, dict):
        NOME_REDE_BNREP = _network_info.get('bnrep_name', _network_info.get('bnrep_id', NOME_REDE_BNREP))
if not isinstance(NOME_REDE_BNREP,str) or not re.fullmatch(r'[A-Za-z0-9_.-]{1,100}',NOME_REDE_BNREP):
    raise ValueError('Nome bnRep inválido.')

EXECUTAR_EXPERIMENTO = True
EXECUTAR_GERACAO = True
SEED = 42
TEMPERATURE = 0.1
MAX_RETRIES = 5

# "auto": use NF4 on GPUs below 32 GiB; BF16/FP16 on larger GPUs.
# To force a mode, use "4bit" or "bf16".
MODO_QUANTIZACAO = os.environ.get('QWEN_QUANTIZACAO', 'auto').strip().lower()
if MODO_QUANTIZACAO not in ('auto','4bit','bf16','fp16','none'):
    raise ValueError('QWEN_QUANTIZACAO deve ser auto, 4bit, bf16, fp16 ou none.')

if MODO_QUANTIZACAO == 'auto':
    USAR_4BITS = GPU_VRAM_GB < 32
elif MODO_QUANTIZACAO == '4bit':
    USAR_4BITS = True
else:
    USAR_4BITS = False

if USAR_4BITS:
    print('[QUANTIZAÇÃO] NF4 4 bits ativado para reduzir uso de VRAM.')
    print('[ATENÇÃO METODOLÓGICA] Se sua execução anterior usou BF16/FP16, registre a mudança de precisão.')
else:
    print('[QUANTIZAÇÃO] Desativada: execução em BF16/FP16 conforme a GPU.')

MAX_NEW_TOKENS_FULLDIST = 2048
MAX_NEW_TOKENS_FULLDIST_RETRY = 4096
MAX_NEW_TOKENS_CONFIANCA = 1536
MAX_RETRIES_CONFIANCA = 4

LIMIAR_CONFIANCA = float(os.environ.get('LIMIAR_CONFIANCA','0.80'))
if not 0 <= LIMIAR_CONFIANCA <= 1:
    raise ValueError('LIMIAR_CONFIANCA precisa estar entre 0 e 1.')

RN_SIGMA_DEFAULT = 0.16
RN_SIGMA_MIN = 0.03
RN_SIGMA_MAX = 0.45
RN_REGULARIZACAO_PESOS = 0.02
RN_ENUMERAR_DIRECOES_ATE = 6
RN_EPS = 1e-12

ASSUMIR_ORDEM_ESTADOS_SEM_METADADO = False

REPETICOES = 1
SOMENTE_RECOMBINAR = False
EXIGIR_TRES_PERSONAS = True
METODO_HIBRIDO = 'ConfidenceRankedNodes'

ID_SESSAO = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESULTADOS = BASE_DIR/'resultados_qwen35_9b_confidence_rn'
RESULTADOS.mkdir(parents=True,exist_ok=True)
ID_JSON = hashlib.sha256(CAMINHO_JSON.read_bytes()).hexdigest()[:12] if CAMINHO_JSON.is_file() else 'sem_json'
MODEL_SLUG = re.sub(r'[^A-Za-z0-9_.-]+','_',HF_MODEL_ID)[:110]
DIR_ESTUDO = RESULTADOS/f'{NOME_REDE_BNREP}_{ID_JSON}'/MODEL_SLUG
DIR_ESTUDO.mkdir(parents=True,exist_ok=True)
CAMINHO_BIF_GOLD = None
REALIZAR_INFERENCIA = EXECUTAR_EXPERIMENTO and not SOMENTE_RECOMBINAR

print('AMBIENTE: Google Colab',flush=True)
print('ENTRADAS: rede =', CAMINHO_JSON,'| personas =',CAMINHO_PERSONAS,flush=True)
print('REFERÊNCIA: NÃO É CONSULTADA NESTE NOTEBOOK | nome somente para auditoria:',NOME_REDE_BNREP,flush=True)
print('MODELO:',HF_MODEL_ID,'| revisão:',HF_MODEL_REVISION,flush=True)
print('PROTOCOLO:',METODO_HIBRIDO,'| confiança >=',LIMIAR_CONFIANCA,'=> FullDist; abaixo => Ranked Nodes',flush=True)
print('RESULTADOS PERSISTENTES:',DIR_ESTUDO,flush=True)
print('HF_HOME TEMPORÁRIO:',os.getenv('HF_HOME','padrão'),flush=True)

assert HF_MODEL_ID == 'Qwen/Qwen3.5-9B'
assert QWEN35_SEM_THINKING is True

## 1. Qwen 3.5 9B loader — a single GPU load

Uses the official multimodal loader, with text-only inputs and `enable_thinking=False`. Text responses are generated justifications, not access to hidden internal reasoning.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
import networkx as nx
import matplotlib.pyplot as plt

from jinja2.exceptions import TemplateError
from pgmpy.factors.discrete import TabularCPD
try:
    from pgmpy.models import DiscreteBayesianNetwork as BayesianNetworkModel
except ImportError:
    from pgmpy.models import BayesianNetwork as BayesianNetworkModel

_TOKENIZER = None
_PROCESSOR = None
_MODEL = None
_GENERATOR = None
_CHAT_TEMPLATE_MODE = 'qwen35_multimodal_text_only'
_SYSTEM_FALLBACK_WARNED = False
_ULTIMA_GERACAO_META = {}


def load_qwen_model():
    """Historical LLMAPI loader interface. Loads ONLY Qwen3.5-9B (once)."""
    global _TOKENIZER, _PROCESSOR, _MODEL, _GENERATOR
    if _MODEL is not None:
        return _MODEL
    import torch
    from transformers import AutoTokenizer, AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
    from transformers import logging as hf_logging
    hf_logging.set_verbosity_warning()
    if not torch.cuda.is_available():
        raise RuntimeError('GPU CUDA necessária. No Colab, selecione um ambiente de execução com GPU.')
    print('[QWEN3.5-9B] Carregando:', HF_MODEL_ID,flush=True)
    dev=torch.cuda.get_device_properties(0)
    print('[GPU]',dev.name,'| VRAM: %.1f GiB' % (dev.total_memory/2**30),flush=True)
    major,_=torch.cuda.get_device_capability(0)
    dtype=torch.bfloat16 if major>=8 else torch.float16
    if dev.total_memory < 24*(2**30) and not USAR_4BITS:
        print('[AVISO] Menos de 24 GiB e BF16/FP16 sem quantização: risco elevado de OOM. '
              'Considere A100 40GB ou NF4 explícito.',flush=True)
    quant=(BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
           bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
           if USAR_4BITS else None)
    token=os.getenv('HF_TOKEN') or None
    torch.manual_seed(SEED)
    start_time=time.perf_counter()
    print('[QWEN3.5-9B] Obtendo tokenizer e processor (cache ou download)...',flush=True)
    _TOKENIZER=AutoTokenizer.from_pretrained(HF_MODEL_ID,revision=HF_MODEL_REVISION,token=token)
    _PROCESSOR=AutoProcessor.from_pretrained(HF_MODEL_ID,revision=HF_MODEL_REVISION,token=token)
    print('[QWEN3.5-9B] Obtendo pesos; isso pode levar alguns minutos...',flush=True)
    try:
        opts=dict(revision=HF_MODEL_REVISION,token=token,device_map='auto',
                  dtype=dtype,low_cpu_mem_usage=True)
        if quant is not None:opts['quantization_config']=quant
        _MODEL=AutoModelForMultimodalLM.from_pretrained(HF_MODEL_ID,**opts)
        _MODEL.eval()
    except Exception as exc:
        print('[FALHA QWEN] Verifique versão recente de Transformers, CUDA, VRAM '
              'e compatibilidade da quantização NF4:',repr(exc),flush=True)
        raise
    _GENERATOR=_MODEL
    print('[QWEN3.5-9B] Modelo pronto em %.1fs | dtype=%s | NF4=%s' %
          (time.perf_counter()-start_time,dtype,USAR_4BITS),flush=True)
    return _MODEL


def _sample(system_prompt: str, user_prompt: str,
            temperature: float=TEMPERATURE, max_new_tokens: int=MAX_NEW_TOKENS_FULLDIST) -> str:
    """Text-only generation with the multimodal model; retain the complete text for auditing."""
    global _CHAT_TEMPLATE_MODE, _SYSTEM_FALLBACK_WARNED, _ULTIMA_GERACAO_META
    import torch
    model=load_qwen_model()
    messages=[]
    if system_prompt:
        messages.append({'role':'system','content':[{'type':'text','text':system_prompt}]})
    messages.append({'role':'user','content':[{'type':'text','text':user_prompt}]})
    args=dict(tokenize=True,add_generation_prompt=True,return_dict=True,
              return_tensors='pt',enable_thinking=not QWEN35_SEM_THINKING)
    try:
        inputs=_PROCESSOR.apply_chat_template(messages,**args)
        _CHAT_TEMPLATE_MODE='processor_text_only_nonthinking'
    except (ValueError,TypeError,TemplateError) as exc:
        # Some template versions do not accept a separate system-role block.
        # Preserve the persona and keep the non-thinking option explicit.
        if not system_prompt:
            raise RuntimeError('Chat template Qwen3.5 incompatível; atualize Transformers.') from exc
        if not _SYSTEM_FALLBACK_WARNED:
            print('[AVISO] Template não aceitou papel system; movendo suas instruções para user:',
                  repr(exc),flush=True)
            _SYSTEM_FALLBACK_WARNED=True
        messages=[{'role':'user','content':[{'type':'text','text':
             f'INSTRUCTIONS:\n{system_prompt}\n\nTASK:\n{user_prompt}'}]}]
        inputs=_PROCESSOR.apply_chat_template(messages,**args)
        _CHAT_TEMPLATE_MODE='processor_system_text_in_user_nonthinking'
    if 'input_ids' not in inputs:
        raise RuntimeError('Processor Qwen3.5 não retornou input_ids para o prompt textual.')
    device=model.device
    inputs={k:(v.to(device) if hasattr(v,'to') else v) for k,v in inputs.items()}
    prompt_len=int(inputs['input_ids'].shape[-1])
    eos=getattr(model.generation_config,'eos_token_id',None) or _TOKENIZER.eos_token_id
    kw=dict(max_new_tokens=int(max_new_tokens),do_sample=bool(temperature>0),
            pad_token_id=_TOKENIZER.pad_token_id or _TOKENIZER.eos_token_id,
            eos_token_id=eos)
    if temperature>0:
        kw.update(temperature=temperature,top_p=1.0)
    start_time=time.perf_counter()
    with torch.inference_mode():
        ids=model.generate(**inputs,**kw)
    new_token_ids=ids[0][prompt_len:]
    full=_PROCESSOR.decode(new_token_ids,skip_special_tokens=True,
                            clean_up_tokenization_spaces=False).strip()
    _ULTIMA_GERACAO_META={'prompt_tokens':prompt_len,'tokens_gerados':int(new_token_ids.shape[-1]),
        'tempo_inferencia_segundos':round(time.perf_counter()-start_time,3),
        'thinking_habilitado':not QWEN35_SEM_THINKING}
    return full


class LLMAPI:
    call_counter = 0
    log_path = None  # initialized after loading personas and creating DIR_EXECUCAO

    @staticmethod
    def call(system_prompt: str, user_prompt: str,
             temperature: float=TEMPERATURE, max_retries: int=MAX_RETRIES,
             max_new_tokens: int=MAX_NEW_TOKENS_FULLDIST, context=None) -> str:
        if not REALIZAR_INFERENCIA:
            raise RuntimeError('A execução está bloqueada. Configure REALIZAR_INFERENCIA=True na seção 0.')
        LLMAPI.call_counter += 1
        call_id = LLMAPI.call_counter
        print(f'    [CONSULTA {call_id:05d}] Enviando ao modelo...', flush=True)
        last_exc = None
        context = context or {}
        for attempt in range(1, max_retries + 1):
            start_time = time.perf_counter()
            try:
                # Use the same seed for the same question/repetition across conditions.
                # Resumed checkpoints do not change the random sequence of subsequent questions.
                import torch
                material=json.dumps({'prompt':user_prompt,'rep':context.get('rep',0),
                     'parse_tentativa':context.get('parse_tentativa',1),
                     'row_retry':context.get('row_retry',0),
                     'chamada_tentativa':attempt},
                     sort_keys=True,ensure_ascii=False)
                seed_call=(SEED+int(hashlib.sha256(material.encode()).hexdigest()[:8],16))%(2**32)
                torch.manual_seed(seed_call)
                if torch.cuda.is_available():torch.cuda.manual_seed_all(seed_call)
                response = _sample(system_prompt, user_prompt,
                                   temperature=temperature, max_new_tokens=max_new_tokens)
                audit = {'utc':datetime.now(timezone.utc).isoformat(), 'call_id':call_id,
                         'attempt':attempt, 'model':HF_MODEL_ID,
                         'session_id':ID_SESSAO,
                         'temperature':temperature, 'seed_consulta':seed_call,
                         'revisao_solicitada':HF_MODEL_REVISION,
                         'qwen35_sem_thinking':QWEN35_SEM_THINKING,
                         'estrategia_chat_template':_CHAT_TEMPLATE_MODE,
                         'context':context, 'system_prompt':system_prompt, 'response':response,
                         'prompt':user_prompt, 'seconds':time.perf_counter()-start_time,
                         'geracao_meta':dict(_ULTIMA_GERACAO_META)}
                with LLMAPI.log_path.open('a', encoding='utf-8') as f:
                    f.write(json.dumps(audit, ensure_ascii=False)+'\n')
                response_excerpt = ' '.join(response.split())
                print(f'    [CONSULTA {call_id:05d}] OK, {audit["seconds"]:.1f}s | '
                      f'resposta: {response_excerpt[-220:]}', flush=True)
                return response
            except (KeyboardInterrupt, SystemExit):
                raise
            except Exception as exc:
                last_exc = exc
                print(f'[WARN] Chamada {call_id} tentativa {attempt}/{max_retries}: {exc}')
                if attempt < max_retries:
                    time.sleep(min(2**attempt, 15))
        raise RuntimeError(f'Falha na chamada ao modelo {call_id}: {last_exc}')

## 2. Read the supplied network (without consulting the reference)


In [ ]:
# ==========================================================
# NODE INPUT
# ==========================================================
def load_bn_spec(json_path: str) -> Tuple[Dict[str, Dict[str, Any]], List[Tuple[str, str]]]:
    """
    Read a JSON file containing:
    - network: general network metadata, optional
    - nodes: nodes in either the legacy or new format
    - edges: causal relationships already defined by the user

    Accepted legacy format:
    {
      "nodes": {
        "A": {
          "description": "Description of node A.",
          "states": ["False", "True"]
        }
      },
      "edges": [...]
    }

    Accepted new format:
    {
      "network": {...},
      "nodes": [
        {
          "name": "A",
          "definition": "...",
          "entity": "...",
          "scale_type": "ordinal",
          "states": [
            {"name": "Low", "meaning": "..."},
            {"name": "High", "meaning": "..."}
          ],
          "temporal_context": "...",
          "measurement_context": "...",
          "notes": "..."
        }
      ],
      "edges": [...]
    }
    """
    path = Path(json_path)

    # utf-8-sig handles input files that contain a BOM
    with path.open("r", encoding="utf-8-sig") as f:
        data = json.load(f)

    raw_nodes = data.get("nodes")
    raw_edges = data.get("edges", [])
    network_info = data.get("network", {})

    if raw_nodes is None:
        raise ValueError("O JSON precisa conter o campo 'nodes'.")

    nodes: Dict[str, Dict[str, Any]] = {}

    # ======================================================
    # CASE 1: legacy format
    # "nodes": { "A": {"description": "...", "states": [...] } }
    # ======================================================
    if isinstance(raw_nodes, dict):
        for name, value in raw_nodes.items():
            name = str(name).strip()

            if isinstance(value, str):
                nodes[name] = {
                    "description": value.strip(),
                    "states": ["False", "True"],
                }

            elif isinstance(value, dict):
                description = str(value.get("description", name)).strip()
                states = value.get("states", ["False", "True"])

                if not isinstance(states, list) or len(states) < 2:
                    raise ValueError(f"O nó '{name}' precisa ter pelo menos dois estados.")

                nodes[name] = {
                    "description": description,
                    "states": [str(s).strip() for s in states],
                }

            else:
                raise ValueError(f"Formato inválido para o nó '{name}'.")

    # ======================================================
    # CASE 2: new format
    # "nodes": [ {"name": "...", "definition": "...", ... } ]
    # ======================================================
    elif isinstance(raw_nodes, list):
        for node_obj in raw_nodes:
            if not isinstance(node_obj, dict):
                raise ValueError(f"Cada item de 'nodes' precisa ser um objeto. Recebido: {node_obj}")

            name = str(node_obj.get("name", "")).strip()
            if not name:
                raise ValueError(f"Nó sem campo 'name': {node_obj}")

            raw_states = node_obj.get("states", [])
            if not isinstance(raw_states, list) or len(raw_states) < 2:
                raise ValueError(f"O nó '{name}' precisa ter pelo menos dois estados.")

            state_names = []
            state_descriptions = []

            for state in raw_states:
                if isinstance(state, dict):
                    state_name = str(state.get("name", "")).strip()
                    state_meaning = str(state.get("meaning", "")).strip()

                    if not state_name:
                        raise ValueError(f"Estado sem campo 'name' no nó '{name}': {state}")

                    state_names.append(state_name)

                    if state_meaning:
                        state_descriptions.append(f"{state_name}: {state_meaning}")

                else:
                    state_name = str(state).strip()
                    state_names.append(state_name)

            description_parts = []

            network_domain = str(network_info.get("domain", "")).strip()
            network_description = str(network_info.get("description", "")).strip()

            if network_domain:
                description_parts.append(f"Network domain: {network_domain}.")

            if network_description:
                description_parts.append(f"Network description: {network_description}")

            definition = str(node_obj.get("definition", "")).strip()
            entity = str(node_obj.get("entity", "")).strip()
            scale_type = str(node_obj.get("scale_type", "")).strip()
            temporal_context = str(node_obj.get("temporal_context", "")).strip()
            measurement_context = str(node_obj.get("measurement_context", "")).strip()
            notes = str(node_obj.get("notes", "")).strip()

            if definition:
                description_parts.append(f"Definition: {definition}")

            if entity:
                description_parts.append(f"Entity: {entity}")

            if scale_type:
                description_parts.append(f"Scale type: {scale_type}")

            if state_descriptions:
                description_parts.append("State meanings: " + "; ".join(state_descriptions))

            if temporal_context:
                description_parts.append(f"Temporal context: {temporal_context}")

            if measurement_context:
                description_parts.append(f"Measurement context: {measurement_context}")

            if notes:
                description_parts.append(f"Notes: {notes}")

            description = " ".join(description_parts).strip()

            if not description:
                description = name

            nodes[name] = {
                "description": description,
                "states": state_names,
                "raw_metadata": node_obj,
            }

    else:
        raise ValueError("'nodes' deve ser um objeto ou uma lista.")

    edges = parse_and_validate_edges(raw_edges, nodes)

    return nodes, edges

def parse_and_validate_edges(
    raw_edges: List[Any],
    nodes_info: Dict[str, Dict[str, Any]],
) -> List[Tuple[str, str]]:
    """
    Convert and validate the edges supplied in the JSON.

    Accepts:
    - {"source": "A", "target": "B"}
    - ["A", "B"]

    Returns:
    - [("A", "B"), ...]
    """
    known_nodes = set(nodes_info.keys())
    edges: List[Tuple[str, str]] = []

    if not isinstance(raw_edges, list):
        raise ValueError("'edges' deve ser uma lista.")

    for e in raw_edges:
        if isinstance(e, dict):
            src = str(e.get("source", "")).strip()
            dst = str(e.get("target", "")).strip()

        elif isinstance(e, (list, tuple)) and len(e) >= 2:
            src = str(e[0]).strip()
            dst = str(e[1]).strip()

        else:
            raise ValueError(f"Aresta inválida: {e}")

        if src not in known_nodes:
            raise ValueError(f"Aresta inválida: nó de origem desconhecido: {src}")

        if dst not in known_nodes:
            raise ValueError(f"Aresta inválida: nó de destino desconhecido: {dst}")

        if src == dst:
            raise ValueError(f"Aresta inválida: self-loop detectado: {src} -> {dst}")

        edges.append((src, dst))

    # Remove duplicates while preserving order
    deduped = []
    seen = set()

    for edge in edges:
        if edge not in seen:
            seen.add(edge)
            deduped.append(edge)

    # Check that the graph is a DAG
    graph = nx.DiGraph()
    graph.add_nodes_from(nodes_info.keys())
    graph.add_edges_from(deduped)

    if not nx.is_directed_acyclic_graph(graph):
        raise ValueError("As arestas fornecidas geram ciclo. A Rede Bayesiana precisa ser um DAG.")

    return deduped


def node_context_text(
    nodes_info: Dict[str, Dict[str, Any]],
    only_nodes: Optional[List[str]] = None
) -> str:
    selected = only_nodes if only_nodes is not None else list(nodes_info.keys())

    lines = []

    for node in selected:
        info = nodes_info[node]

        states = ", ".join(info["states"])

        lines.append(
            f"- {node}: {info['description']} Possible states: [{states}]"
        )

    return "\n".join(lines)

## 3. Read the three researcher-defined personas

The `personas.json` file is still used. In this version, the notebook requires **exactly three active personas** and runs four independent conditions: `controle`, `persona 1`, `persona 2`, and `persona 3`. The weights in the JSON are recorded but do not change the individual generation of each network.


In [ ]:
# ============================================================
# 3.1 — Personas: exactly 3 active personas + control
# ============================================================
PERSONAS_CONFIG = None
PERSONAS_ATIVAS = []
PERSONAS_HASH = None

def load_personas_json(path, nodes_info):
    path=Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'JSON de personas não encontrado: {path}.')
    payload=json.loads(path.read_text(encoding='utf-8-sig'))
    if not isinstance(payload,dict) or not isinstance(payload.get('personas'),list):
        raise ValueError('personas.json precisa conter a lista "personas".')

    ids=set()
    personas=[]
    for p in payload['personas']:
        if not isinstance(p,dict):
            raise ValueError('Toda persona precisa ser um objeto JSON.')
        pid=p.get('id','')
        if not isinstance(pid,str) or not re.fullmatch(r'[A-Za-z0-9_-]{1,60}',pid):
            raise ValueError(f'ID de persona inválido: {pid!r}.')
        if pid.lower() in ('controle','ensemble','uniform','random') or pid in ids:
            raise ValueError('ID de persona repetido/reservado: '+pid)
        ids.add(pid)
        if not isinstance(p.get('ativo',True),bool):
            raise ValueError('"ativo" deve ser booleano em '+pid)
        if not p.get('ativo',True):
            continue

        role_description=p.get('papel','')
        specialty=p.get('especialidade','')
        instructions=p.get('instrucoes','')
        if not all(isinstance(x,str) for x in (role_description,specialty,instructions)):
            raise ValueError('papel, especialidade e instrucoes devem ser strings em '+pid)
        if not role_description.strip() or not instructions.strip():
            raise ValueError('Persona ativa precisa de papel e instrucoes: '+pid)

        w=p.get('peso',0.0)
        if isinstance(w,bool) or not isinstance(w,(int,float)) or not np.isfinite(w) or w<0:
            raise ValueError('peso inválido em '+pid)

        personas.append({
            'id':pid,
            'ativo':True,
            'papel':role_description.strip(),
            'especialidade':specialty.strip(),
            'instrucoes':instructions.strip(),
            'peso':float(w),
        })

    if EXIGIR_TRES_PERSONAS and len(personas)!=3:
        raise ValueError(f'Este protocolo exige exatamente 3 personas ativas; encontrado: {len(personas)}.')
    if not personas:
        raise ValueError('Ao menos uma persona ativa é necessária.')

    control_config=payload.get('controle',{'ativo':True})
    if not isinstance(control_config,dict) or not isinstance(control_config.get('ativo',True),bool):
        raise ValueError('controle precisa de campo "ativo" booleano.')
    if not control_config.get('ativo',True):
        raise ValueError('Este protocolo exige controle.ativo=true.')

    calibration_config=payload.get('calibracao',{})
    if not isinstance(calibration_config,dict):
        raise ValueError('calibracao deve ser um objeto.')
    repetition_count=calibration_config.get('repeticoes',1)
    if type(repetition_count) is not int or not 1<=repetition_count<=20:
        raise ValueError('calibracao.repeticoes deve ser inteiro de 1 a 20.')
    recombine_only=calibration_config.get('somente_recombinar',False)
    if not isinstance(recombine_only,bool):
        raise ValueError('calibracao.somente_recombinar deve ser booleano.')

    legacy_methods=calibration_config.get('metodos',[])
    node_weights=calibration_config.get('pesos_por_no',{})
    if node_weights and not isinstance(node_weights,dict):
        raise ValueError('pesos_por_no deve ser objeto.')

    print('[PERSONAS] 3 personas ativas + controle:',flush=True)
    for p in personas:
        print(f"  {p['id']:22s} peso_documental={p['peso']:.4f} | papel={p['papel']}",flush=True)
    if legacy_methods:
        print('[INFO] calibracao.metodos do JSON é ignorado nesta versão. Método fixo:',METODO_HIBRIDO,flush=True)
    print('[INFO] Pesos de persona são somente metadados; nenhuma fusão é feita na geração.',flush=True)

    return {
        'personas':personas,
        'controle':True,
        'repeticoes':repetition_count,
        'pesos_por_no':node_weights,
        'somente_recombinar':recombine_only,
        'hash':hashlib.sha256(path.read_bytes()).hexdigest(),
        'arquivo':str(path.resolve()),
    }


def persona_system_prompt(p):
    if p is None:
        return ''
    parts=[f"You are acting in the role of: {p['papel']}."]
    if p.get('especialidade'):
        parts.append(f"Specialty: {p['especialidade']}.")
    parts.append('Additional persona instructions supplied by the researcher: '+p['instrucoes'])
    parts.append('Follow the exact output format requested by the user prompt. Report uncertainty honestly.')
    return '\n'.join(parts)


## 4. Prompts: confidence screening + FullDist

For each node with parents, Qwen receives **a single screening call** containing all parent configurations. In this call, it provides no CPT probabilities: it only reports its confidence in estimating each row and the expected qualitative direction of each parent (`positive`, `negative`, or `uncertain`).

Only rows with `confidence >= LIMIAR_CONFIANCA` receive a FullDist call. These distributions are the anchors used to fit Ranked Nodes.


In [ ]:
# ----------------------------------------------------------
# PROMPTS AND PARSERS — confidence + FullDist
# ----------------------------------------------------------
def _relevant_descriptions(nodes_info, node, parents):
    return node_context_text(nodes_info, [node] + list(parents))


def _parent_condition(parents, parent_state_values):
    return (', '.join(f'{p} is {v}' for p, v in zip(parents, parent_state_values))
            if parents else 'there are no parent conditions')


def format_epk_prompt_full_distribution(nodes_info, node, parents, parent_state_values):
    # Retain the previous notebook's FullDist template for anchor comparability.
    desc = _relevant_descriptions(nodes_info, node, parents)
    order = ', '.join(nodes_info[node]['states'])
    question = (f'Given that {_parent_condition(parents, parent_state_values)}, '
                f'what is the probability distribution of {node}?')
    return f"""These nodes are related to the question inside a Bayesian Network:

{desc}

Given this information, answer the following question by providing the probability distribution of the node. Your answer should include your reasoning and at the end a sentence that says 'The probability distribution of the node is:' followed by the probabilities given in a tuple with each probability representing a state in the given order.

Order of states: ({order})

{question}"""


def format_node_confidence_prompt(nodes_info, node, parents, parent_combinations):
    desc=_relevant_descriptions(nodes_info,node,parents)
    row_entries='\n'.join(
        f'{i}: '+', '.join(f'{p}={v}' for p,v in zip(parents,comb))
        for i,comb in enumerate(parent_combinations)
    )
    parent_names=', '.join(parents)
    return f"""These variables belong to a Bayesian Network:

{desc}

Before estimating any numerical CPT probabilities, assess whether your INTERNAL KNOWLEDGE is sufficient to make an informed conditional probability estimate for each parent configuration of node {node}.

Important definition of confidence:
- confidence measures how much relevant knowledge you believe you have for estimating the FULL probability distribution;
- it does NOT mean how easy it is to invent a plausible numeric answer;
- use values from 0 to 1;
- do not provide any CPT probabilities in this response.

Also report the qualitative influence of each parent on higher states of {node}: "positive", "negative", or "uncertain". This direction is only a qualitative aid for Ranked Nodes.

Parents: {parent_names}
Configurations:
{row_entries}

Return ONLY one valid JSON object in exactly this structure:
{{
  "parent_influence": [
    {{"parent": "PARENT_NAME", "direction": "positive|negative|uncertain", "confidence": 0.0}}
  ],
  "rows": [
    {{"row": 0, "confidence": 0.0}}
  ]
}}

Include every parent exactly once and every row index from 0 to {len(parent_combinations)-1} exactly once. Do not include explanations outside the JSON."""


_NUMERO_PROB = (r'(?P<num>(?:\d+(?:\.\d*)?|\.\d+)'
                r'(?:\s*/\s*(?:\d+(?:\.\d*)?|\.\d+))?)'
                r'(?P<percent>\s*%)?(?![\w/])')


def _parse_probability_number(match):
    parsed_value = match.group('num').strip()
    if '/' in parsed_value:
        a,b=parsed_value.split('/',1)
        den=float(b.strip())
        if den==0:
            raise ValueError('Fração com denominador zero.')
        prob=float(a.strip())/den
    else:
        prob=float(parsed_value)
    if match.group('percent'):
        prob/=100
    if not np.isfinite(prob) or not 0<=prob<=1:
        raise ValueError('Probabilidade fora de [0,1]: '+str(prob))
    return float(prob)


def extract_distribution(response_text, n_states, state_labels=None):
    def validate_probability_tuple(tuple_content):
        parts=[p.strip() for p in tuple_content.split(',')]
        if len(parts)!=n_states:
            return None
        parsed_output=[]
        for p in parts:
            num=re.fullmatch(_NUMERO_PROB,p)
            if num is None:
                return None
            parsed_output.append(_parse_probability_number(num))
        return parsed_output if sum(parsed_output)>0 else None

    distribution_markers=[
        r'the\s+probability\s+distribution\s+of\s+the\s+node\s+is\s*[:=]',
        r'the\s+(?:final|estimated)\s+probability\s+distribution\s+is\s*[:=]',
        r'(?:final\s+)?probabilities\s+in\s+order\s*[:=]',
        r'final\s+distribution\s*[:=]',
    ]
    marker_matches=[mt for pattern in distribution_markers for mt in re.finditer(pattern,response_text,re.I)]
    if marker_matches:
        conclusion_text=response_text[max(marker_matches,key=lambda mt:mt.start()).end():]
        probability_tuple=re.search(r'[\(\[]\s*([^\)\]]+?)\s*[\)\]]',conclusion_text)
        if probability_tuple and probability_tuple.start()<240:
            parsed_values=validate_probability_tuple(probability_tuple.group(1))
            if parsed_values is not None:
                return parsed_values

    row_entries=[row_entry.strip() for row_entry in response_text.splitlines() if row_entry.strip()]
    if row_entries:
        last_line=re.sub(r'^\s*(?:final answer|answer)\s*[:=]\s*','',row_entries[-1],flags=re.I)
        mt=re.fullmatch(r'[\(\[]\s*([^\)\]]+)\s*[\)\]]\s*[.\s]*',last_line)
        if mt:
            parsed_values=validate_probability_tuple(mt.group(1))
            if parsed_values is not None:
                return parsed_values

    if state_labels and len(state_labels)==n_states:
        text_block=response_text[-600:].strip().rstrip('.')
        num=r'(?:\d+(?:\.\d*)?|\.\d+)(?:\s*/\s*(?:\d+(?:\.\d*)?|\.\d+))?\s*%?'
        pattern=r'^\s*'+r'\s*[,;\n]\s*'.join(
            rf'{re.escape(s)}\s*[:=]\s*({num})' for s in state_labels)+r'\s*$'
        row_entries=text_block.splitlines()
        text_candidates=[text_block,'\n'.join(row_entries[-n_states:]),row_entries[-1]]
        for text_candidate in text_candidates:
            text_candidate=re.sub(r'^\s*(?:final\s+distribution|probabilities\s+in\s+order)\s*[:=]\s*',
                              '',text_candidate,flags=re.I)
            mt=re.fullmatch(pattern,text_candidate,flags=re.I)
            if mt:
                nums=[]
                for matched_field in mt.groups():
                    v=re.fullmatch(_NUMERO_PROB,matched_field.strip())
                    if v is None:
                        break
                    nums.append(_parse_probability_number(v))
                if len(nums)==n_states and sum(nums)>0:
                    return nums

    raise ValueError('FullDist: distribuição final ambígua; não extrair valores intermediários.')


def normalize_distribution(raw_probs):
    arr=np.asarray(raw_probs,dtype=float)
    if arr.ndim!=1 or arr.size<2 or np.any(~np.isfinite(arr)) or np.any(arr<0):
        raise ValueError('Distribuição inválida.')
    s=arr.sum()
    if s<=0:
        raise ValueError('Probabilidades todas zero.')
    return (arr/s).tolist()


def _extract_json_object(response_text):
    response_text=response_text.strip()
    response_text=re.sub(r'^```(?:json)?\s*','',response_text,flags=re.I)
    response_text=re.sub(r'\s*```$','',response_text)
    try:
        return json.loads(response_text)
    except Exception:
        start_time=response_text.find('{')
        end_index=response_text.rfind('}')
        if start_time>=0 and end_index>start_time:
            return json.loads(response_text[start_time:end_index+1])
        raise ValueError('Nenhum objeto JSON válido encontrado.')


def extract_confidences(response_text, parents, n_rows):
    obj=_extract_json_object(response_text)
    if not isinstance(obj,dict):
        raise ValueError('Resposta de confiança precisa ser objeto JSON.')
    rows=obj.get('rows')
    influences=obj.get('parent_influence')
    if not isinstance(rows,list) or not isinstance(influences,list):
        raise ValueError('JSON precisa conter rows e parent_influence.')

    row_confidences={}
    for x in rows:
        if not isinstance(x,dict) or type(x.get('row')) is not int:
            raise ValueError('Cada row precisa de índice inteiro.')
        idx=x['row']
        c=x.get('confidence')
        if idx in row_confidences or idx<0 or idx>=n_rows:
            raise ValueError('Índice de row inválido/repetido: '+str(idx))
        if isinstance(c,bool) or not isinstance(c,(int,float)) or not np.isfinite(c) or not 0<=c<=1:
            raise ValueError('Confiança inválida na row '+str(idx))
        row_confidences[idx]=float(c)
    if set(row_confidences)!=set(range(n_rows)):
        raise ValueError('A resposta não contém exatamente todas as linhas esperadas.')

    infl={}
    for x in influences:
        if not isinstance(x,dict):
            raise ValueError('parent_influence inválido.')
        p=x.get('parent')
        d=str(x.get('direction','')).lower().strip()
        c=x.get('confidence')
        if p not in parents or p in infl:
            raise ValueError('Pai ausente/desconhecido/repetido: '+str(p))
        if d not in ('positive','negative','uncertain'):
            raise ValueError('Direção inválida em '+str(p))
        if isinstance(c,bool) or not isinstance(c,(int,float)) or not np.isfinite(c) or not 0<=c<=1:
            raise ValueError('Confiança de direção inválida em '+str(p))
        infl[p]={'direction':d,'confidence':float(c)}
    if set(infl)!=set(parents):
        raise ValueError('parent_influence precisa conter todos os pais exatamente uma vez.')

    return row_confidences,infl


def log_parser_event(method_name, parser_context, parse_attempt, answer, accepted,
                             parsed_value=None, extraction_rule=None, error_value=None):
    parser_context=dict(parser_context or {})
    parser_event={
        'utc':datetime.now(timezone.utc).isoformat(),
        'session_id':ID_SESSAO,
        'metodo':method_name,
        'persona_id':parser_context.get('persona_id','controle'),
        'repeticao':parser_context.get('rep',0)+1,
        'no':parser_context.get('node'),
        'pais':parser_context.get('pais'),
        'tentativa_parser':parse_attempt,
        'aceita':bool(accepted),
        'valor':parsed_value,
        'regra_extracao':extraction_rule,
        'erro':str(error_value) if error_value is not None else None,
        'resposta_integral':answer,
    }
    record_path=globals().get('PARSER_LOG_PATH')
    if record_path is not None:
        Path(record_path).parent.mkdir(parents=True,exist_ok=True)
        with Path(record_path).open('a',encoding='utf-8') as f:
            f.write(json.dumps(parser_event,ensure_ascii=False,default=str)+'\n')
    return parser_event


def query_llm_full_distribution(prompt,n_states,temperature=TEMPERATURE,
                                verbose=False,max_retries=MAX_RETRIES,system_prompt='',context=None):
    context=dict(context or {})
    for parse_attempt in range(1,max_retries+1):
        raw=LLMAPI.call(
            system_prompt=system_prompt,
            user_prompt=prompt,
            temperature=temperature,
            max_new_tokens=MAX_NEW_TOKENS_FULLDIST if parse_attempt==1 else MAX_NEW_TOKENS_FULLDIST_RETRY,
            context={**context,'parse_tentativa':parse_attempt},
        )
        try:
            arr=extract_distribution(raw,n_states,state_labels=context.get('states'))
            arr=normalize_distribution(arr)
            log_parser_event('FullDist',context,parse_attempt,raw,True,arr,'distribuicao_final')
            print(f'    [EXTRAÇÃO FullDist] {context.get("node","?")}: {arr}',flush=True)
            if verbose:
                print('PROMPT:',prompt,'\nRESPOSTA:',raw,'\nvalores:',arr)
            return arr,raw
        except ValueError as exc:
            log_parser_event('FullDist',context,parse_attempt,raw,False,error_value=exc)
            print(f'[WARN] FullDist tentativa {parse_attempt}/{max_retries}: {exc}',flush=True)
            if parse_attempt==max_retries:
                raise


def query_llm_confidence(prompt,parents,n_rows,system_prompt='',context=None):
    context=dict(context or {})
    for parse_attempt in range(1,MAX_RETRIES_CONFIANCA+1):
        raw=LLMAPI.call(
            system_prompt=system_prompt,
            user_prompt=prompt,
            temperature=0.0,
            max_new_tokens=MAX_NEW_TOKENS_CONFIANCA,
            context={**context,'parse_tentativa':parse_attempt,'fase':'triagem_confianca'},
        )
        try:
            conf,infl=extract_confidences(raw,parents,n_rows)
            log_parser_event(
                'ConfidenceScreen',
                context,
                parse_attempt,
                raw,
                True,
                {'rows':conf,'parent_influence':infl},
                'json_estruturado',
            )
            return conf,infl,raw
        except (ValueError,json.JSONDecodeError) as exc:
            log_parser_event('ConfidenceScreen',context,parse_attempt,raw,False,error_value=exc)
            print(f'[WARN] Confiança tentativa {parse_attempt}/{MAX_RETRIES_CONFIANCA}: {exc}',flush=True)
            if parse_attempt==MAX_RETRIES_CONFIANCA:
                raise


## 5. Hybrid CPT construction

The Ranked Nodes method used here is a **WMEAN** implementation. Each ordinal state occupies an equally spaced interval in `[0,1]`. The parents are combined through a weighted mean, with either positive or negative directions. A Normal distribution truncated to `[0,1]` converts the mean into probabilities for the child node's states.

The weights and dispersion parameter `sigma` are fitted to approximate the FullDist distributions of high-confidence rows, using **confidence-weighted squared Hellinger distance**. When few anchors are available, fitting uses regularization and the qualitative directions reported during screening. No reference CPT is used in this fit.


In [ ]:
# ----------------------------------------------------------
# RANKED NODES + HYBRID GENERATION
# ----------------------------------------------------------
from scipy.optimize import minimize
from scipy.special import ndtr


def _scale_type_ordinal(info):
    meta=info.get('raw_metadata',{}) if isinstance(info,dict) else {}
    st=str(meta.get('scale_type','')).strip().lower() if isinstance(meta,dict) else ''
    if st in ('ordinal','ordered','ranked'):
        return True
    return bool(ASSUMIR_ORDEM_ESTADOS_SEM_METADADO)


def _rn_eligible(nodes_info,node,parents):
    if not parents:
        return False
    return _scale_type_ordinal(nodes_info[node]) and all(_scale_type_ordinal(nodes_info[p]) for p in parents)


def _state_midpoint(nodes_info,node,state):
    state_labels=nodes_info[node]['states']
    i=state_labels.index(state)
    k=len(state_labels)
    return (i+0.5)/k


def _rn_probs(mu,sigma,k):
    sigma=float(np.clip(sigma,RN_SIGMA_MIN,RN_SIGMA_MAX))
    mu=float(np.clip(mu,0.0,1.0))
    bounds=np.linspace(0.0,1.0,k+1)
    den=ndtr((1.0-mu)/sigma)-ndtr((0.0-mu)/sigma)
    if not np.isfinite(den) or den<=RN_EPS:
        arr=np.ones(k)/k
        return arr

    vals=[]
    for a,b in zip(bounds[:-1],bounds[1:]):
        vals.append((ndtr((b-mu)/sigma)-ndtr((a-mu)/sigma))/den)

    arr=np.asarray(vals,dtype=float)
    arr=np.clip(arr,RN_EPS,None)
    arr/=arr.sum()
    return arr


def _softmax(z):
    z=np.asarray(z,dtype=float)
    z=z-z.max()
    e=np.exp(z)
    return e/e.sum()


def _hellinger2(p,q):
    p=np.asarray(p,dtype=float)
    q=np.asarray(q,dtype=float)
    return float(0.5*np.sum((np.sqrt(np.clip(p,0,None))-np.sqrt(np.clip(q,0,None)))**2))


def _direction_from_hint(parent_influences,parent):
    d=parent_influences.get(parent,{}).get('direction','uncertain')
    return -1 if d=='negative' else 1


def _row_mean(nodes_info,parents,comb,weights,directions):
    xs=[]
    for p,st,d in zip(parents,comb,directions):
        x=_state_midpoint(nodes_info,p,st)
        xs.append(x if d>0 else 1.0-x)
    return float(np.dot(weights,np.asarray(xs,dtype=float)))


def fit_ranked_nodes(nodes_info,node,parents,pcombs,anchors,parent_influences):
    p=len(parents)
    k=len(nodes_info[node]['states'])
    if p==0:
        raise ValueError('Ranked Nodes exige ao menos um pai.')

    hints=tuple(_direction_from_hint(parent_influences,parent_name) for parent_name in parents)

    # No anchors: use only qualitative screening information and equal weights.
    if not anchors:
        return {
            'weights':[1.0/p]*p,
            'directions':list(hints),
            'sigma':RN_SIGMA_DEFAULT,
            'loss':None,
            'fit_status':'sem_ancoras_pesos_iguais_direcao_triagem',
            'n_anchors':0,
        }

    # With few anchors, retain the reported directions.
    # With enough anchors, test possible directions and let the anchors determine the fit.
    min_anchors_for_direction=max(2,p+1)
    if len(anchors)>=min_anchors_for_direction and p<=RN_ENUMERAR_DIRECOES_ATE:
        direction_candidates=list(itertools.product((1,-1),repeat=p))
    else:
        direction_candidates=[hints]

    conf=np.asarray([max(float(a['confidence']),RN_EPS) for a in anchors],dtype=float)
    conf=conf/conf.sum()
    best_fit=None
    log_lo,log_hi=np.log(RN_SIGMA_MIN),np.log(RN_SIGMA_MAX)

    for directions in direction_candidates:
        def objective(theta):
            weights=_softmax(theta[:p])
            sigma=float(np.exp(theta[p]))
            error_value=0.0
            for w_anchor,a in zip(conf,anchors):
                mu=_row_mean(nodes_info,parents,a['comb'],weights,directions)
                pred=_rn_probs(mu,sigma,k)
                error_value+=float(w_anchor)*_hellinger2(a['distribution'],pred)
            reg=RN_REGULARIZACAO_PESOS*float(np.sum((weights-(1.0/p))**2))
            return error_value+reg

        x0=np.r_[np.zeros(p),np.log(RN_SIGMA_DEFAULT)]
        bounds=[(-6,6)]*p+[(log_lo,log_hi)]
        res=minimize(objective,x0,method='L-BFGS-B',bounds=bounds,options={'maxiter':500})

        rec={
            'weights':_softmax(res.x[:p]).tolist(),
            'directions':list(directions),
            'sigma':float(np.exp(res.x[p])),
            'loss':float(res.fun),
            'fit_status':'ajustado' if res.success else 'ajustado_com_aviso_'+str(res.message),
            'n_anchors':len(anchors),
        }
        if best_fit is None or rec['loss']<best_fit['loss']-1e-12:
            best_fit=rec

    return best_fit


def _hybrid_fingerprint(nodes_info,edges,persona,repetition):
    b=json.dumps({
        'nodes':nodes_info,
        'edges':edges,
        'modelo':HF_MODEL_ID,
        'revision':HF_MODEL_REVISION,
        'temperature':TEMPERATURE,
        '4bits':USAR_4BITS,
        'seed':SEED,
        'metodo':METODO_HIBRIDO,
        'limiar_confianca':LIMIAR_CONFIANCA,
        'rn_sigma_default':RN_SIGMA_DEFAULT,
        'rn_sigma_min':RN_SIGMA_MIN,
        'rn_sigma_max':RN_SIGMA_MAX,
        'rn_reg':RN_REGULARIZACAO_PESOS,
        'persona':({k:persona.get(k) for k in ('id','papel','especialidade','instrucoes')} if persona else None),
        'repeticao':repetition,
        'schema':'confidence_rn_v1',
    },sort_keys=True,ensure_ascii=False,default=str).encode()
    return hashlib.sha256(b).hexdigest()[:16]


def build_empty_network(nodes_info,edges):
    model=BayesianNetworkModel(edges)
    model.add_nodes_from(nodes_info.keys())
    return model


def generate_confidence_rn_cpts(model,nodes_info,persona=None,repetition=0,checkpoint_dir=None,verbose=True):
    global PARSER_LOG_PATH

    condition_label=persona['id'] if persona else 'controle'
    system_prompt=persona_system_prompt(persona)
    checkpoint_dir=Path(checkpoint_dir or DIR_ESTUDO/'checkpoints_confidence_rn')
    checkpoint_dir.mkdir(parents=True,exist_ok=True)

    fingerprint=_hybrid_fingerprint(nodes_info,list(model.edges()),persona,repetition)
    audit_dir=DIR_ESTUDO/'auditoria_confidence_rn'
    audit_dir.mkdir(parents=True,exist_ok=True)
    LLMAPI.log_path=audit_dir/f'consultas_{fingerprint}.jsonl'
    PARSER_LOG_PATH=audit_dir/f'parser_{fingerprint}.jsonl'

    details_dir=checkpoint_dir/fingerprint
    details_dir.mkdir(parents=True,exist_ok=True)

    cpds=[]
    metadata={
        'condicao':condition_label,
        'metodo':METODO_HIBRIDO,
        'limiar_confianca':LIMIAR_CONFIANCA,
        'nodes':{},
        'fingerprint':fingerprint,
    }
    start_time=time.perf_counter()

    for node_index,node in enumerate(model.nodes(),start=1):
        parents=list(model.get_parents(node))
        states=nodes_info[node]['states']
        pcombs=list(itertools.product(*[nodes_info[p]['states'] for p in parents])) if parents else [()]
        ordinal=_rn_eligible(nodes_info,node,parents)

        print(
            f'\n[NÓ {node_index}/{len(model.nodes())}] {node} | '
            f'pais={parents or "nenhum"} | linhas={len(pcombs)} | RN={ordinal}',
            flush=True,
        )

        node_meta={
            'parents':parents,
            'states':states,
            'ordinal_rn_elegivel':ordinal,
            'rows':[],
        }

        # Root node: a single distribution; RN does not reduce the cost here.
        if not parents:
            file_path=details_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_root.json'
            if file_path.is_file():
                reg=json.loads(file_path.read_text(encoding='utf-8'))
                dist=reg['distribution']
                print('  [CHECKPOINT ROOT]',dist,flush=True)
            else:
                prompt=format_epk_prompt_full_distribution(nodes_info,node,parents,())
                dist,raw=query_llm_full_distribution(
                    prompt,
                    len(states),
                    system_prompt=system_prompt,
                    context={
                        'persona_id':condition_label,
                        'rep':repetition,
                        'node':node,
                        'pais':{},
                        'states':states,
                        'fase':'root_fullDist',
                    },
                )
                reg={
                    'node':node,
                    'row':0,
                    'parents':{},
                    'confidence':None,
                    'source':'LLM_FULLDIST_ROOT',
                    'distribution':dist,
                    'raw':raw,
                    'prompt':prompt,
                    'utc':datetime.now(timezone.utc).isoformat(),
                }
                _save_json_atomic(file_path,reg)

            row_entries=[dist]
            node_meta['rows']=[{
                'row':0,
                'parents':{},
                'confidence':None,
                'source':'LLM_FULLDIST_ROOT',
                'distribution':dist,
            }]
            node_meta['rn_parameters']=None

        # Non-ordinal node: do not silently apply Ranked Nodes.
        elif not ordinal:
            row_entries=[]
            print('  [FALLBACK] Nó/pais não marcados como ordinal; FullDist em todas as linhas.',flush=True)

            for ci,comb in enumerate(pcombs):
                file_path=details_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_fallback_{ci}.json'
                if file_path.is_file():
                    reg=json.loads(file_path.read_text(encoding='utf-8'))
                    dist=reg['distribution']
                else:
                    prompt=format_epk_prompt_full_distribution(nodes_info,node,parents,comb)
                    dist,raw=query_llm_full_distribution(
                        prompt,
                        len(states),
                        system_prompt=system_prompt,
                        context={
                            'persona_id':condition_label,
                            'rep':repetition,
                            'node':node,
                            'pais':dict(zip(parents,comb)),
                            'states':states,
                            'fase':'nonordinal_fullDist',
                        },
                    )
                    reg={
                        'node':node,
                        'row':ci,
                        'parents':dict(zip(parents,comb)),
                        'confidence':None,
                        'source':'LLM_FULLDIST_FALLBACK_NONORDINAL',
                        'distribution':dist,
                        'raw':raw,
                        'prompt':prompt,
                        'utc':datetime.now(timezone.utc).isoformat(),
                    }
                    _save_json_atomic(file_path,reg)

                row_entries.append(dist)
                node_meta['rows'].append({
                    'row':ci,
                    'parents':dict(zip(parents,comb)),
                    'confidence':None,
                    'source':'LLM_FULLDIST_FALLBACK_NONORDINAL',
                    'distribution':dist,
                })

            node_meta['rn_parameters']=None

        else:
            # 1) One confidence screening call for all configurations of the node.
            conf_file=details_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_confidence.json'

            if conf_file.is_file():
                cdata=json.loads(conf_file.read_text(encoding='utf-8'))
                conf={int(k):float(v) for k,v in cdata['row_confidence'].items()}
                parent_influences=cdata['parent_influence']
                print('  [CHECKPOINT CONFIANÇA] reutilizado.',flush=True)
            else:
                pconf=format_node_confidence_prompt(nodes_info,node,parents,pcombs)
                conf,parent_influences,rawc=query_llm_confidence(
                    pconf,
                    parents,
                    len(pcombs),
                    system_prompt=system_prompt,
                    context={
                        'persona_id':condition_label,
                        'rep':repetition,
                        'node':node,
                        'pais':parents,
                    },
                )
                cdata={
                    'node':node,
                    'parents':parents,
                    'row_confidence':{str(k):v for k,v in conf.items()},
                    'parent_influence':parent_influences,
                    'raw':rawc,
                    'prompt':pconf,
                    'limiar':LIMIAR_CONFIANCA,
                    'utc':datetime.now(timezone.utc).isoformat(),
                }
                _save_json_atomic(conf_file,cdata)

            selected_rows=[i for i in range(len(pcombs)) if conf[i]>=LIMIAR_CONFIANCA]
            print(
                f'  [CONFIANÇA] {len(selected_rows)}/{len(pcombs)} linhas '
                f'>= {LIMIAR_CONFIANCA:.2f}: {selected_rows}',
                flush=True,
            )

            # 2) FullDist only for high-confidence rows.
            anchors=[]
            dist_anchors={}

            for ci in selected_rows:
                comb=pcombs[ci]
                node_checkpoint_path=details_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_anchor_{ci}.json'

                if node_checkpoint_path.is_file():
                    reg=json.loads(node_checkpoint_path.read_text(encoding='utf-8'))
                    dist=reg['distribution']
                    print(f'    [CHECKPOINT ÂNCORA {ci}]',dist,flush=True)
                else:
                    prompt=format_epk_prompt_full_distribution(nodes_info,node,parents,comb)
                    dist,raw=query_llm_full_distribution(
                        prompt,
                        len(states),
                        system_prompt=system_prompt,
                        context={
                            'persona_id':condition_label,
                            'rep':repetition,
                            'node':node,
                            'pais':dict(zip(parents,comb)),
                            'states':states,
                            'fase':'anchor_fullDist',
                            'confidence':conf[ci],
                        },
                    )
                    reg={
                        'node':node,
                        'row':ci,
                        'parents':dict(zip(parents,comb)),
                        'confidence':conf[ci],
                        'source':'LLM_FULLDIST',
                        'distribution':dist,
                        'raw':raw,
                        'prompt':prompt,
                        'utc':datetime.now(timezone.utc).isoformat(),
                    }
                    _save_json_atomic(node_checkpoint_path,reg)

                dist_anchors[ci]=dist
                anchors.append({
                    'row':ci,
                    'comb':comb,
                    'confidence':conf[ci],
                    'distribution':dist,
                })

            # 3) Fit RN only to the anchors; the gold reference is never used here.
            params=fit_ranked_nodes(
                nodes_info,
                node,
                parents,
                pcombs,
                anchors,
                parent_influences,
            )
            params.update({
                'parent_names':parents,
                'parent_influence_llm':parent_influences,
                'anchor_rows':selected_rows,
                'threshold':LIMIAR_CONFIANCA,
            })

            _save_json_atomic(
                details_dir/f'{re.sub("[^A-Za-z0-9_-]","_",node)}_rn_fit.json',
                params,
            )

            print(
                '  [RN]',
                {k:params[k] for k in ('weights','directions','sigma','loss','fit_status','n_anchors')},
                flush=True,
            )

            # 4) Final CPT: anchors retain LLM distributions; the remaining rows come from RN.
            row_entries=[]
            for ci,comb in enumerate(pcombs):
                if ci in dist_anchors:
                    dist=dist_anchors[ci]
                    source='LLM_FULLDIST'
                else:
                    mu=_row_mean(
                        nodes_info,
                        parents,
                        comb,
                        params['weights'],
                        params['directions'],
                    )
                    dist=_rn_probs(mu,params['sigma'],len(states)).tolist()
                    source='RANKED_NODES'

                row_entries.append(dist)
                node_meta['rows'].append({
                    'row':ci,
                    'parents':dict(zip(parents,comb)),
                    'confidence':conf[ci],
                    'source':source,
                    'distribution':dist,
                })

            node_meta['rn_parameters']=params
            node_meta['confidence_summary']={
                'min':float(min(conf.values())),
                'max':float(max(conf.values())),
                'mean':float(np.mean(list(conf.values()))),
                'n_high_confidence':len(selected_rows),
                'n_ranked_nodes':len(pcombs)-len(selected_rows),
            }

        kwargs=dict(
            variable=node,
            variable_card=len(states),
            values=np.asarray(row_entries,dtype=float).T,
            state_names={node:states,**{p:nodes_info[p]['states'] for p in parents}},
        )
        if parents:
            kwargs.update(
                evidence=parents,
                evidence_card=[len(nodes_info[p]['states']) for p in parents],
            )

        cpds.append(TabularCPD(**kwargs))
        metadata['nodes'][node]=node_meta
        print(f'[NÓ CONCLUÍDO] {node} | {len(cpds)}/{len(model.nodes())} CPTs',flush=True)

    metadata['duration_seconds']=round(time.perf_counter()-start_time,3)
    metadata['call_counter_total_session']=LLMAPI.call_counter
    return cpds,metadata


def estimate_confidence_rn_network(nodes_info,edges,persona=None,repetition=0,checkpoint_dir=None,verbose=True):
    network_model=build_empty_network(nodes_info,edges)
    cpds,meta=generate_confidence_rn_cpts(
        network_model,
        nodes_info,
        persona=persona,
        repetition=repetition,
        checkpoint_dir=checkpoint_dir,
        verbose=verbose,
    )
    network_model.add_cpds(*cpds)
    assert network_model.check_model(), 'pgmpy: estrutura/CPT híbrida inválida'
    return network_model,meta


In [ ]:
def export_result(model, nodes_info, output_path='resultado_rede_hibrida.json', hybrid_metadata=None):
    data={
        'nodes':nodes_info,
        'edges':[{'source':src,'target':dst} for src,dst in model.edges()],
        'cpds':{},
        'generation_method':METODO_HIBRIDO,
    }

    if hybrid_metadata is not None:
        data['hybrid_metadata']=hybrid_metadata

    for cpd in model.get_cpds():
        data['cpds'][cpd.variable]={
            'variables':list(cpd.variables),
            'state_names':cpd.state_names,
            'values':cpd.get_values().tolist(),
        }

    with open(output_path,'w',encoding='utf-8') as f:
        json.dump(data,f,ensure_ascii=False,indent=2)

    print(f'\n[INFO] Resultado híbrido exportado para: {output_path}')


## 6. Initial query: does the model report familiarity with the network?


In [ ]:
# Diagnostic step: one query per model/network/session, without a persona or any gold CPT.
FAMILIARIDADE_REDE = None
FAMILIARIDADE_VERSAO = 'v1_apenas_nome_sem_topologia_antes_das_CPTs'

def _network_familiarity_prompt(item_name):
    familiarity_system_prompt = (
        'You are participating in a scientific study of Bayesian network parameter elicitation. '
        'Report uncertainty accurately. Do not pretend to remember a benchmark, graph or CPT '
        'when you only recognize its general topic. Do not invent numerical probabilities. '
        'You have no external tools in this query.'
    )
    question = (
        f'Before being shown any graph, node descriptions, states, data, or probability tables: '
        f'were you ALREADY familiar with the SPECIFIC named Bayesian network benchmark "{item_name}"? '
        'Distinguish clearly: (a) knowledge of its general domain, (b) recognition of this exact '
        'benchmark/network by name, (c) prior knowledge of its original nodes and edges, and '
        '(d) prior knowledge of its original numerical CPTs. Answer yes, no, or uncertain '
        'for each aspect and briefly explain what you know, or say you do not know. '
        'Do not guess CPT numbers or claim that the generic domain is the named benchmark.'
    )
    return familiarity_system_prompt, question

def query_network_familiarity():
    global FAMILIARIDADE_REDE
    record_path=DIR_EXECUCAO/'familiaridade_rede.json'
    response_text=DIR_EXECUCAO/'familiaridade_rede.log'
    original_hash=hashlib.sha256(CAMINHO_JSON.read_bytes()).hexdigest()
    if SOMENTE_RECOMBINAR or not PERGUNTAR_FAMILIARIDADE_REDE:
        if record_path.is_file():
            previous_record=json.loads(record_path.read_text(encoding='utf-8'))
            if (previous_record.get('modelo')==HF_MODEL_ID and
                previous_record.get('sha256_json_rede')==original_hash and
                previous_record.get('nome_rede')==NOME_REDE_BNREP):
                FAMILIARIDADE_REDE={**previous_record,'registro_reutilizado':True}
                print('[FAMILIARIDADE] Registro anterior compatível reutilizado:', record_path,flush=True)
                return FAMILIARIDADE_REDE
        reason=('somente_recombinar' if SOMENTE_RECOMBINAR else 'desativada_explicitamente')
        FAMILIARIDADE_REDE={'status':'nao_consultada','motivo':reason,
                           'nome_rede':NOME_REDE_BNREP,'modelo':HF_MODEL_ID,
                           'sha256_json_rede':original_hash,'resposta_completa':None,
                           'autorrelato_nao_comprovacao':True}
        print('[FAMILIARIDADE] Consulta não executada:', reason,flush=True)
        return FAMILIARIDADE_REDE
    familiarity_system_prompt,question=_network_familiarity_prompt(NOME_REDE_BNREP)
    print('\n'+'='*78, '\n[PRIMEIRA CONSULTA] Conhecimento prévio declarado sobre a rede:',NOME_REDE_BNREP,flush=True)
    print('[FAMILIARIDADE] Apenas o nome da rede; sem nós, descrições, arestas, CPTs e sem persona.',flush=True)
    previous_log_path=LLMAPI.log_path
    LLMAPI.log_path=DIR_EXECUCAO/'familiaridade_rede_chamadas.jsonl'
    start_time=time.perf_counter()
    try:
        answer=LLMAPI.call(familiarity_system_prompt,question,temperature=0.0,max_retries=2,
                            max_new_tokens=384,
                            context={'metodo':'FamiliaridadeRede','persona_id':'sem_persona',
                                     'rep':0,'nome_rede':NOME_REDE_BNREP})
    finally:
        LLMAPI.log_path=previous_log_path
    familiarity_record={
        'status':'respondida', 'versao_pergunta':FAMILIARIDADE_VERSAO,
        'utc':datetime.now(timezone.utc).isoformat(),'id_sessao':ID_SESSAO,
        'modelo':HF_MODEL_ID,'revisao_solicitada':HF_MODEL_REVISION,
        'qwen35_sem_thinking':QWEN35_SEM_THINKING,
        'nome_rede':NOME_REDE_BNREP,'sha256_json_rede':original_hash,
        'sem_persona':True,'antes_da_elicitacao':True,'gold_bnrep_ainda_nao_consultada':True,
        'pergunta_sistema':familiarity_system_prompt,'pergunta_usuario':question,'resposta_completa':answer,
        'duracao_segundos':round(time.perf_counter()-start_time,3),
        'temperatura':0.0,'tokens_maximos':384,
        'caminho_chamadas':str(DIR_EXECUCAO/'familiaridade_rede_chamadas.jsonl'),
        'autorrelato_nao_comprovacao':True,
        'resposta_nao_injetada_nos_prompts_das_CPTs':True
    }
    # Save before starting any method, including if the runtime stops after the query.
    _save_json_atomic(record_path,familiarity_record)
    response_text.write_text('[PERGUNTA INICIAL - FAMILIARIDADE]\n'+question+
                     '\n\n[RESPOSTA COMPLETA]\n'+answer+'\n',encoding='utf-8')
    FAMILIARIDADE_REDE=familiarity_record
    print('[FAMILIARIDADE] Resposta completa (autorrelato; NÃO comprova memorização):\n'+answer,flush=True)
    print('[FAMILIARIDADE] Arquivos:',record_path,'|',response_text,flush=True)
    print('[FAMILIARIDADE] Resposta guardada apenas como diagnóstico; não altera as CPTs.',flush=True)
    return familiarity_record

## 7. Run generation — control + 3 personas

This cell creates **four complete, independent networks**:

- control without a persona;
- structural engineer;
- seismologist;
- risk analyst.

In each condition, the protocol is `Confidence screening -> FullDist for anchors -> Ranked Nodes for the remaining rows`. bnRep remains outside this stage.


In [ ]:
import contextlib, shutil, sys, traceback
from datetime import datetime,timezone

SCHEMA_GERACAO = 'qwen35_confidence_ranked_nodes_v1'
DIR_GERACAO = None


def file_sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def _save_json_atomic(path, content):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(
        json.dumps(content,ensure_ascii=False,indent=2,default=str),
        encoding='utf-8',
    )
    temp.replace(path)


class Tee:
    def __init__(self,path):
        self.path=Path(path)
        self.path.parent.mkdir(parents=True,exist_ok=True)
        self.file=None
        self.console=None

    def __enter__(self):
        self.console=sys.stdout
        self.file=self.path.open('a',encoding='utf-8',buffering=1)
        sys.stdout=self
        return self

    def __exit__(self,*args):
        sys.stdout=self.console
        self.file.close()

    def write(self,text):
        self.console.write(text)
        self.file.write(text)

    def flush(self):
        self.console.flush()
        self.file.flush()


def validate_generation_inputs():
    global PERSONAS_CONFIG,PERSONAS_ATIVAS,PERSONAS_HASH
    global REPETICOES,SOMENTE_RECOMBINAR,REALIZAR_INFERENCIA,DIR_EXECUCAO,DIR_GERACAO

    if not CAMINHO_JSON.is_file() or not CAMINHO_PERSONAS.is_file():
        raise FileNotFoundError(
            'Envie manualmente os dois JSONs para /content e confira CAMINHO_REDE_MANUAL e CAMINHO_PERSONAS_MANUAL.'
        )

    network_nodes,network_edges=load_bn_spec(str(CAMINHO_JSON))
    PERSONAS_CONFIG=load_personas_json(CAMINHO_PERSONAS,network_nodes)
    PERSONAS_ATIVAS=PERSONAS_CONFIG['personas']
    PERSONAS_HASH=PERSONAS_CONFIG['hash']
    REPETICOES=PERSONAS_CONFIG['repeticoes']

    if PERSONAS_CONFIG['somente_recombinar']:
        raise ValueError('Desative calibracao.somente_recombinar: este notebook gera CPTs.')

    SOMENTE_RECOMBINAR=False
    REALIZAR_INFERENCIA=True

    config_material={
        'schema':SCHEMA_GERACAO,
        'json_rede_sha256':file_sha256(CAMINHO_JSON),
        'json_personas_sha256':file_sha256(CAMINHO_PERSONAS),
        'persona_ids':[p['id'] for p in PERSONAS_ATIVAS],
        'model':HF_MODEL_ID,
        'revision':HF_MODEL_REVISION,
        '4bits':USAR_4BITS,
        'seed':SEED,
        'temperature':TEMPERATURE,
        'repeticoes':REPETICOES,
        'metodo':METODO_HIBRIDO,
        'limiar_confianca':LIMIAR_CONFIANCA,
        'rn_sigma_default':RN_SIGMA_DEFAULT,
        'rn_sigma_bounds':[RN_SIGMA_MIN,RN_SIGMA_MAX],
        'rn_regularizacao_pesos':RN_REGULARIZACAO_PESOS,
        'qwen35_sem_thinking':QWEN35_SEM_THINKING,
    }

    cid=hashlib.sha256(
        json.dumps(config_material,sort_keys=True).encode()
    ).hexdigest()[:16]

    DIR_GERACAO=DIR_ESTUDO/'geracoes'/cid
    DIR_GERACAO.mkdir(parents=True,exist_ok=True)
    DIR_EXECUCAO=DIR_GERACAO

    for item_name,source_path in [
        ('rede_original.json',CAMINHO_JSON),
        ('personas_original.json',CAMINHO_PERSONAS),
    ]:
        destination_path=DIR_GERACAO/item_name
        if destination_path.exists() and file_sha256(destination_path)!=file_sha256(source_path):
            raise RuntimeError(f'Entrada congelada difere: {destination_path}')
        if not destination_path.exists():
            shutil.copy2(source_path,destination_path)

    execution_plan=[]
    conditions=[('controle',None)]+[(p['id'],p) for p in PERSONAS_ATIVAS]

    for rep in range(REPETICOES):
        for cond,persona in conditions:
            execution_plan.append({
                'condicao':cond,
                'persona_id':persona['id'] if persona else None,
                'repeticao':rep+1,
                'metodo':METODO_HIBRIDO,
                'cpt_file':f'cpts/{cond}_{METODO_HIBRIDO}_r{rep+1}.json',
                'metadata_file':f'detalhes/{cond}_{METODO_HIBRIDO}_r{rep+1}.json',
                'console_log':f'logs/{cond}_{METODO_HIBRIDO}_r{rep+1}.log',
                'status':'pendente',
            })

    manifest_path=DIR_GERACAO/'manifesto_geracao.json'

    if manifest_path.is_file():
        existing_manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
        if existing_manifest.get('configuracao')!=config_material:
            raise RuntimeError('Manifesto existente incompatível.')

        previous_runs={
            (x['condicao'],x['repeticao']):x
            for x in existing_manifest['execucoes']
        }
        for x in execution_plan:
            run_key=(x['condicao'],x['repeticao'])
            if run_key in previous_runs:
                x.update(previous_runs[run_key])
        created_at=existing_manifest.get('criado_em_utc')
    else:
        created_at=datetime.now(timezone.utc).isoformat()

    payload={
        'schema':SCHEMA_GERACAO,
        'status':'em_andamento',
        'criado_em_utc':created_at,
        'atualizado_em_utc':datetime.now(timezone.utc).isoformat(),
        'configuracao':config_material,
        'nome_rede_bnrep_apenas_identificador':NOME_REDE_BNREP,
        'sem_consulta_ao_bnrep':True,
        'sem_dados_observados':True,
        'modelo':HF_MODEL_ID,
        'json_rede':'rede_original.json',
        'json_personas':'personas_original.json',
        'pesos_personas_nao_aplicados_na_geracao':True,
        'personas_executadas':[p['id'] for p in PERSONAS_ATIVAS],
        'familiaridade_arquivo':'familiaridade_rede.json' if PERGUNTAR_FAMILIARIDADE_REDE else None,
        'execucoes':execution_plan,
    }
    _save_json_atomic(manifest_path,payload)

    # Estimate the cost before knowing how many rows will exceed the threshold.
    g=nx.DiGraph()
    g.add_nodes_from(network_nodes)
    g.add_edges_from(network_edges)

    roots=[n for n in network_nodes if g.in_degree(n)==0]
    rn_nodes=[
        n for n in network_nodes
        if g.in_degree(n)>0 and _rn_eligible(network_nodes,n,list(g.predecessors(n)))
    ]

    fallback_rows=sum(
        int(np.prod([len(network_nodes[p]['states']) for p in g.predecessors(n)]))
        for n in network_nodes
        if g.in_degree(n)>0 and n not in rn_nodes
    )

    min_calls_per_condition=len(roots)+len(rn_nodes)+fallback_rows

    full_rows=sum(
        int(np.prod([len(network_nodes[p]['states']) for p in g.predecessors(n)]))
        if g.in_degree(n)>0 else 1
        for n in network_nodes
    )

    print('[VALIDADO]',len(network_nodes),'nós;',len(network_edges),'arestas;',full_rows,'linhas CPT.',flush=True)
    print('[CONDIÇÕES] Controle + 3 personas:',[p['id'] for p in PERSONAS_ATIVAS],flush=True)
    print('[PROTOCOLO] Sem SepState. FullDist somente em raízes/âncoras/fallback não ordinal.',flush=True)
    print(
        '[CHAMADAS BASE POR CONDIÇÃO antes das âncoras]',
        min_calls_per_condition,
        '= raízes',len(roots),
        '+ triagens',len(rn_nodes),
        '+ fallback não ordinal',fallback_rows,
        flush=True,
    )
    print('[OBS] Cada linha que superar o limiar adiciona 1 chamada FullDist.',flush=True)
    print('[RESULTADOS]',DIR_GERACAO,flush=True)

    return network_nodes,network_edges,payload,manifest_path


def run_experiment():
    global FAMILIARIDADE_REDE

    network_nodes,network_edges,manifest,generation_manifest_path=validate_generation_inputs()

    if not EXECUTAR_GERACAO:
        print('[DESATIVADO] Apenas conferência dos JSONs.')
        return manifest

    pending_runs=[
        x for x in manifest['execucoes']
        if x['status']!='concluido' or not (DIR_GERACAO/x['cpt_file']).is_file()
    ]

    familiarity_path=DIR_GERACAO/'familiaridade_rede.json'
    needs_familiarity_check=PERGUNTAR_FAMILIARIDADE_REDE and not familiarity_path.is_file()

    if not pending_runs and not needs_familiarity_check:
        manifest['status']='concluido'
        _save_json_atomic(generation_manifest_path,manifest)
        print('[CONCLUÍDO] Todas as redes já existem.')
        return manifest

    print('[MODELO] Carregando uma única vez. bnRep NÃO será consultado.',flush=True)
    load_qwen_model()

    import torch
    manifest['versao_transformers']=__import__('transformers').__version__
    manifest['modelo_commit_carregado']=getattr(_MODEL.config,'_commit_hash',None)
    manifest['gpu']=torch.cuda.get_device_name(0)
    _save_json_atomic(generation_manifest_path,manifest)

    if PERGUNTAR_FAMILIARIDADE_REDE:
        if familiarity_path.is_file():
            previous_record=json.loads(familiarity_path.read_text(encoding='utf-8'))
            if (
                previous_record.get('modelo')!=HF_MODEL_ID
                or previous_record.get('sha256_json_rede')!=file_sha256(CAMINHO_JSON)
            ):
                raise RuntimeError('Familiaridade anterior refere-se a rede/modelo diferentes.')
            FAMILIARIDADE_REDE={**previous_record,'registro_reutilizado':True}
            print('[FAMILIARIDADE] Registro anterior reutilizado.',flush=True)
        else:
            query_network_familiarity()

    for item in manifest['execucoes']:
        destination_path=DIR_GERACAO/item['cpt_file']
        meta_path=DIR_GERACAO/item['metadata_file']

        if item['status']=='concluido' and destination_path.is_file():
            print('[CONDIÇÃO EXISTENTE]',item['condicao'],'rep',item['repeticao'])
            continue

        cond=item['condicao']
        persona=next((p for p in PERSONAS_ATIVAS if p['id']==cond),None)
        rep=item['repeticao']-1
        log=DIR_GERACAO/item['console_log']

        try:
            with Tee(log):
                print('\n'+'='*78,flush=True)
                print('[CONDIÇÃO]',cond,'|',METODO_HIBRIDO,'| rep',rep+1,flush=True)
                print(
                    '[SEM GOLD] Estrutura + descrições + persona. '
                    'Confiança decide quais linhas recebem FullDist.',
                    flush=True,
                )

                start_time=time.perf_counter()

                network_model,meta=estimate_confidence_rn_network(
                    network_nodes,
                    network_edges,
                    persona=persona,
                    repetition=rep,
                    checkpoint_dir=DIR_ESTUDO/'checkpoints_confidence_rn',
                    verbose=True,
                )

                destination_path.parent.mkdir(parents=True,exist_ok=True)
                meta_path.parent.mkdir(parents=True,exist_ok=True)

                export_result(network_model,network_nodes,str(destination_path),hybrid_metadata=meta)
                _save_json_atomic(meta_path,meta)

                elapsed_seconds=round(time.perf_counter()-start_time,3)
                fingerprint=meta['fingerprint']
                audit_dir=DIR_ESTUDO/'auditoria_confidence_rn'

                for source_path,item_name in [
                    (audit_dir/f'consultas_{fingerprint}.jsonl','chamadas'),
                    (audit_dir/f'parser_{fingerprint}.jsonl','parser'),
                ]:
                    target_path=DIR_GERACAO/'logs'/f'{cond}_{METODO_HIBRIDO}_r{rep+1}_{item_name}.jsonl'
                    target_path.parent.mkdir(parents=True,exist_ok=True)
                    if source_path.is_file():
                        shutil.copy2(source_path,target_path)

                n_llm=sum(
                    1
                    for nm in meta['nodes'].values()
                    for row in nm.get('rows',[])
                    if str(row.get('source','')).startswith('LLM_')
                )
                n_rn=sum(
                    1
                    for nm in meta['nodes'].values()
                    for row in nm.get('rows',[])
                    if row.get('source')=='RANKED_NODES'
                )

                item.update({
                    'status':'concluido',
                    'duracao_segundos':elapsed_seconds,
                    'sha256_cpt':file_sha256(destination_path),
                    'fingerprint_checkpoint':fingerprint,
                    'linhas_llm':n_llm,
                    'linhas_ranked_nodes':n_rn,
                    'concluido_em_utc':datetime.now(timezone.utc).isoformat(),
                })

                print(
                    '[SALVO]',
                    destination_path,
                    '| linhas LLM=',n_llm,
                    '| linhas RN=',n_rn,
                    '|',elapsed_seconds,'s',
                    flush=True,
                )

            manifest['atualizado_em_utc']=datetime.now(timezone.utc).isoformat()
            _save_json_atomic(generation_manifest_path,manifest)

        except BaseException as exc:
            item['status']='interrompido'
            item['erro']=repr(exc)
            manifest['status']='interrompido'
            manifest['atualizado_em_utc']=datetime.now(timezone.utc).isoformat()
            _save_json_atomic(generation_manifest_path,manifest)
            print('[FALHA] Checkpoints preservados; execute novamente para retomar.',flush=True)
            raise

    manifest['status']='concluido'
    manifest['concluido_em_utc']=datetime.now(timezone.utc).isoformat()
    _save_json_atomic(generation_manifest_path,manifest)

    print('\n[GERAÇÃO FINALIZADA]',DIR_GERACAO,flush=True)
    return manifest


In [ ]:
MANIFESTO_GERACAO = run_experiment()

## 8. Inspect the outputs

The table below shows the four conditions. `linhas_llm` reports how many final rows came directly from FullDist; `linhas_ranked_nodes` reports how many were completed by Ranked Nodes. The `detalhes/*.json` files retain confidence scores, anchors, weights, directions, sigma, and the source of each row.


In [ ]:
from IPython.display import display
import pandas as pd

if MANIFESTO_GERACAO:
    cols=[
        'condicao',
        'metodo',
        'repeticao',
        'status',
        'cpt_file',
        'metadata_file',
        'linhas_llm',
        'linhas_ranked_nodes',
        'duracao_segundos',
    ]
    df=pd.DataFrame(MANIFESTO_GERACAO['execucoes'])
    display(df[[c for c in cols if c in df.columns]])
    print('PASTA_GERACAO:',DIR_GERACAO)


## 9. Results in Colab

**Results, manifests, logs, and checkpoints** are stored in:

`/content/Qwen_BN_Confidence_RN/`

They remain available while the Colab runtime is active.

Because this version **does not use Google Drive**, a full runtime reset deletes the local files. At the end, run the cell below to create a `.zip` and download it manually through the Colab **Files** panel.


In [ ]:
# ============================================================
# PACKAGE RESULTS FOR MANUAL DOWNLOAD
# ============================================================
import shutil
from pathlib import Path

if DIR_GERACAO is None or not Path(DIR_GERACAO).is_dir():
    raise RuntimeError('Nenhuma pasta de geração encontrada. Execute primeiro o experimento.')

ARQUIVO_ZIP = shutil.make_archive(
    '/content/resultados_qwen35_9b_confidence_ranked_nodes',
    'zip',
    root_dir=DIR_GERACAO
)

print('ZIP criado:', ARQUIVO_ZIP)
print('Baixe-o manualmente pelo painel Arquivos do Google Colab.')
